# 00 — Data probe: what do our hackathon labels actually measure?

**Roadmap:** Track A → Step 0.2 · **Data:** `old-dep/` only · no API, no GPU, ~1 min on a laptop.

**Why this exists.** The hackathon model was trained on 3,000 articles labelled by **one LLM (Gemini)**, ~96% from **two outlets** (efsyn, protothema). Its reported F1 measured agreement with Gemini, not correctness. The model is gone and we're retraining from zero, so first we check **what the data would teach a new model**.

**What triggered it** (`labeled_dataset.jsonl`):

| Observation | Value |
| --- | --- |
| Labels exactly 0.50 | **47%** (protothema: 73%) |
| Labels ≥ 0.70 (right) | **5%** |
| Variance explained by outlet alone | R² = 0.28 |
| efsyn articles from `/politiki/` | 1,402 / 1,443 (topic imbalance) |

**Experiments**

| | Question | Shows |
| --- | --- | --- |
| E1 | Can the outlet be recognised from text? (E1c: politics only) | Whether a shortcut exists, and how much of it is topic vs framing |
| E2 | What do the labels look like? | Compression at 0.50, missing right side |
| E3 | What does a model trained on them learn? | Stance vs outlet detection |
| E4 | Blind reading of 30 articles | Whether the 0.50 pile is truly neutral |
| E5 | Claude vs Gemini on 20 articles | LLM–LLM agreement |

**Limit:** without human gold labels we can show *symptoms*, not prove Gemini *wrong*. That's the gold set's job (Phase 2).

**Terms used**

| Term | Meaning |
| --- | --- |
| TF-IDF | Turns text into word-weight vectors (frequent in this article, rare overall → high weight) |
| Logistic / ridge regression | Simple linear models: classify (outlet) / predict a number (bias) |
| Cross-validation | Each article is predicted by a model that never saw it |
| Accuracy, F1 | Share correct; F1 balances precision and recall per class |
| R² | Share of label variance explained (0 = nothing, 1 = everything) |
| Spearman ρ | Do predictions rank articles in the same order as labels? (−1…1) |
| MAE | Average absolute error, in label units |
| AUC | How well a score separates two groups (0.5 = chance, 1 = perfect) |
| Krippendorff's α | Agreement between raters, corrected for chance (≥ 0.8 reliable) |
| Bootstrap CI | Range of a statistic over resampled data: how uncertain it is |

Key numbers go into `FINDINGS` → `outputs/00_findings_auto.json`.

## 0. Setup
Auto-detects the repo root; one fixed `SEED` so every split and sample is reproducible.

In [ ]:
import json, re, unicodedata, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, mean_absolute_error, roc_auc_score
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
try:
    display                      # available inside Jupyter
except NameError:                # allows running the cells as a plain script too
    display = print

SEED = 42
ROOT = Path.cwd()
while not (ROOT / "old-dep").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "old-dep"
OUT = ROOT / "research" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

FINDINGS = {}                    # every experiment adds its key numbers here
print("repo root:", ROOT)

## 1. Load the data
- `labeled_dataset.jsonl` (3,000, Gemini labels) → E2–E4
- `scraper/*.json` (~9k unlabelled) → extra text for E1 (outlet recognition needs no labels)
- `claude_labels` / `gemini_labels` (same 20 articles) → E5
- **Excluded:** `synthetic/*`, which is LLM-written and describes the generator, not Greek media.

Derived: `section` (from the URL) to separate outlet from topic; `n_words`, since length is itself a fingerprint.

In [ ]:
def read_any(path):
    path = Path(path)
    if path.suffix == ".jsonl":
        return [json.loads(l) for l in path.open(encoding="utf-8") if l.strip()]
    data = json.load(path.open(encoding="utf-8"))
    return data if isinstance(data, list) else data.get("articles", [])

def section_of(url):
    parts = str(url).split("/")
    return parts[3] if len(parts) > 3 else ""

df = pd.DataFrame(read_any(DATA / "datasets" / "labeled_dataset.jsonl"))
df["bias"] = df["ai_labels"].map(lambda d: d.get("bias") if isinstance(d, dict) else np.nan).astype(float)
df["section"] = df["url"].map(section_of)
df["n_words"] = df["text"].fillna("").str.split().str.len()
df = df.dropna(subset=["bias", "text"]).reset_index(drop=True)

print(f"labelled articles: {len(df)}")
display(df["source"].value_counts())
display(df.groupby("source")["n_words"].median().rename("median words"))

## 2. Masking outlet fingerprints
Removes trivial giveaways (outlet names, "Διαβάστε επίσης", URLs, handles, site menus, embeds) so E1 measures **style**, not name-dropping. It's a v0 mask; E1b shows what it still misses.

In [ ]:
OUTLET_PATTERNS = [
    r"\befsyn(?:\.gr)?\b", r"\bεφ\.?\s?συν\b", r"εφημερ[ιί]δα\s+των\s+συντακτ[ωώ]ν",
    r"\bproto\s?thema(?:\.gr)?\b", r"πρ[ωώ]το\s+θ[εέ]μα",
    r"\biefimerida(?:\.gr)?\b",
    r"\btanea(?:\.gr)?\b",
]
BOILERPLATE_PATTERNS = [
    r"διαβ[αά]στε\s+(?:επ[ιί]σης|ακ[οό]μα|ακ[οό]μη)[^.\n]*",
    r"ακολουθ[ηή]στε\s+(?:μας|το)[^.\n]*",
    r"https?://\S+|www\.\S+",
    r"\S+@\S+\.\S+",
    r"@\w+",
    # embedded video-player / cookie notices left in by the scraper (found by E1b on the first run)
    r"to view this video.*?html5 video",
    r"please enable javascript[^.]*\.?",
    r"[^.]*\bcookies?\b[^.]*\.?",
    # protothema's site navigation menu, glued to the start of every article in the raw scrapes
    r"φωτι[αά]\s+τ[ωώ]ρα\s*-\s*πυρκαγι[εέ]ς\s+σε\s+εξ[εέ]λιξη.*?advertorial",
    # Instagram embed captions
    r"δε[ιί]τε\s+αυτ[ηή]\s+τη\s+δημοσ[ιί]ευση\s+στο\s+instagram\.?(?:\s*η\s+δημοσ[ιί]ευση\s+κοινοποι[ηή]θηκε\s+απ[οό][^.]*\.?)?",
]
_outlet_re = re.compile("|".join(OUTLET_PATTERNS), re.IGNORECASE)
_boiler_re = re.compile("|".join(BOILERPLATE_PATTERNS), re.IGNORECASE)

def mask_outlets(text: str) -> str:
    text = _boiler_re.sub(" ", str(text))
    text = _outlet_re.sub(" [ΜΜΕ] ", text)
    return re.sub(r"\s+", " ", text).strip()

df["text_masked"] = df["text"].map(mask_outlets)
df["title_masked"] = df["title"].fillna("").map(mask_outlets)

hits = df["text"].str.count(_outlet_re.pattern, flags=re.IGNORECASE)
print("articles whose text names an outlet (before masking):")
display(hits.gt(0).groupby(df["source"]).sum())

## E1 — Can the outlet be recognised from text alone?
A model trained on bias takes the easiest signal that predicts the label (*shortcut learning*). If the outlet is readable from the text, "detect outlet → output its average" is that shortcut.

**Method:** TF-IDF + logistic regression, 5-fold cross-validation, on raw / masked / title-only text. If this simple model finds the outlet, any transformer will.

**Read:** judge by **efsyn-vs-protothema accuracy**, since iefimerida (~150 articles) is too small and drags macro-F1 down.

| Masked accuracy | Meaning |
| --- | --- |
| ≥ 0.95 | Shortcut wide open; more outlets per side is mandatory |
| 0.80–0.95 | Strong shortcut; mitigate |
| < 0.80 | Not trivially separable |

In [ ]:
# More text for E1: unlabelled scrapes + labelled set, deduplicated by URL and by exact text.
extra = []
for f in ["scraper/efsyn.json", "scraper/articles.json"]:
    p = DATA / f
    if p.exists():
        extra += read_any(p)
# Labelled rows go FIRST: they were cleaned before labelling, while the raw scrapes of the same URL still
# contain site menus. drop_duplicates keeps the first copy, so this order keeps the clean text.
e1 = pd.concat([df[["source", "url", "title", "text"]], pd.DataFrame(extra)], ignore_index=True)
e1 = e1[e1["source"].isin(["efsyn.gr", "protothema.gr", "iefimerida.gr"])]
e1 = e1.dropna(subset=["text"])
e1 = e1[e1["text"].str.split().str.len() >= 50]               # drop empty or near-empty pages
e1 = e1.drop_duplicates("url").drop_duplicates("text").reset_index(drop=True)
e1["text_masked"] = e1["text"].map(mask_outlets)
e1["title_masked"] = e1["title"].fillna("").map(mask_outlets)
print(f"E1 corpus: {len(e1)} unique articles")
display(e1["source"].value_counts())

In [ ]:
def outlet_classifier():
    return make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50_000,
                        sublinear_tf=True, strip_accents="unicode", lowercase=True),
        LogisticRegression(max_iter=2000, class_weight="balanced", C=4.0),
    )

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
labels = sorted(e1["source"].unique())
e1_results = {}
for name, col in [("raw text", "text"), ("masked text", "text_masked"), ("title only", "title_masked")]:
    pred = cross_val_predict(outlet_classifier(), e1[col], e1["source"], cv=cv, n_jobs=-1)
    per_class = f1_score(e1["source"], pred, average=None, labels=labels)
    two = e1["source"].isin(["efsyn.gr", "protothema.gr"]).to_numpy()
    e1_results[name] = {"accuracy": round(accuracy_score(e1["source"], pred), 3),
                        "macro_f1": round(f1_score(e1["source"], pred, average="macro"), 3),
                        **{f"f1 {l}": round(f, 3) for l, f in zip(labels, per_class)},
                        "accuracy efsyn-vs-protothema": round(accuracy_score(e1["source"][two], pred[two]), 3)}
    if name == "masked text":
        cm = pd.DataFrame(confusion_matrix(e1["source"], pred, labels=labels),
                          index=[f"true {l}" for l in labels], columns=[f"pred {l}" for l in labels])

display(pd.DataFrame(e1_results).T)
print("Confusion matrix (masked text):")
display(cm)
FINDINGS["E1_outlet_recognition"] = e1_results

### E1b — What gives the outlet away?
The top-weighted words show *how* the outlet is recognised. Each kind needs a different fix:

| Kind | Example | Fix |
| --- | --- | --- |
| Leftover page junk | menus, players, "read more" | Scraper cleaning |
| Neutral house style | "σύμφωνα με", punctuation | More outlets per side |
| Wire markers | "ΑΠΕ-ΜΠΕ" | Keep; record as `is_wire` (likely no stance) |
| Political framing | "σκάνδαλο", "κυβέρνηση Μητσοτάκη" | Legitimate stance signal; learn it from the article |

> **First-run lesson:** protothema's top words were *league, chef, survivor*, its **site menu**, glued by the old scraper to every raw article. That inflated accuracy to 99.8%. Now fixed (clean copies kept, menu masked). **Scraper requirement:** fail a test when one phrase appears in a large share of an outlet's articles.

In [ ]:
clf = outlet_classifier().fit(e1["text_masked"], e1["source"])
vec, lr = clf.named_steps["tfidfvectorizer"], clf.named_steps["logisticregression"]
vocab = np.array(vec.get_feature_names_out())
top = {cls: vocab[np.argsort(lr.coef_[i])[::-1][:25]] for i, cls in enumerate(lr.classes_)}
display(pd.DataFrame(top))

### E1c — Politics articles only
E1 mixes topic with outlet: protothema covers greece/world/economy, efsyn mostly politics. Keeping only political-section articles, **balanced** (chance = 50%), isolates **how each outlet writes about the same kind of story**. What remains is mostly framing and sourcing. A big drop from E1 means much of the shortcut was topic, so the fix is a balanced section mix per outlet.

Small sample (~136 per outlet, ±5%).

In [ ]:
two = df[df["source"].isin(["efsyn.gr", "protothema.gr"])]
pol = two[two["section"].isin(["politiki", "politics"])]
n_bal = pol["source"].value_counts().min()
pol = pol.groupby("source").sample(n_bal, random_state=SEED)

from sklearn.model_selection import cross_val_score
e1c = {"n_per_outlet": int(n_bal),
       "text_accuracy": round(cross_val_score(outlet_classifier(), pol["text_masked"], pol["source"], cv=cv).mean(), 3),
       "title_accuracy": round(cross_val_score(outlet_classifier(), pol["title_masked"], pol["source"], cv=cv).mean(), 3)}
display(pd.Series(e1c, name="politics only, balanced (chance = 0.5)"))

m = outlet_classifier().fit(pol["text_masked"], pol["source"])
v = np.array(m.named_steps["tfidfvectorizer"].get_feature_names_out())
c = m.named_steps["logisticregression"].coef_[0]          # binary: positive = second class (protothema)
display(pd.DataFrame({"efsyn": v[np.argsort(c)[:30]], "protothema": v[np.argsort(c)[::-1][:30]]}))
FINDINGS["E1c_politics_only"] = e1c

## E2 — What do the labels look like?
A model can only learn the scale it's shown. If half the targets are 0.50 and the right is empty, "predict neutral" becomes the safe answer: the hackathon model's behaviour.

**Computed:** per-outlet percentiles and shares (= 0.50, ≤ 0.30, ≥ 0.70); **R²** of outlet and outlet + section; **round-number heaping** (multiples of 0.05), a sign the LLM has no stable scale, which is why the roadmap prefers pairwise comparison.

In [ ]:
def pct(s, q): return s.quantile(q)
desc = df.groupby("source")["bias"].agg(
    n="count", mean="mean", std="std",
    p10=lambda s: pct(s, .10), p25=lambda s: pct(s, .25), median="median",
    p75=lambda s: pct(s, .75), p90=lambda s: pct(s, .90),
    share_eq_050=lambda s: (s.round(3) == 0.5).mean(),
    share_le_030=lambda s: (s <= 0.3).mean(),
    share_ge_070=lambda s: (s >= 0.7).mean(),
).round(3)
display(desc)

overall = {
    "n": len(df),
    "share_exactly_0.50": round((df["bias"].round(3) == 0.5).mean(), 3),
    "share_ge_0.70": round((df["bias"] >= 0.7).mean(), 3),
    "share_le_0.30": round((df["bias"] <= 0.3).mean(), 3),
    "n_distinct_values": int(df["bias"].round(3).nunique()),
    "share_multiple_of_0.05": round((np.isclose((df["bias"] * 20) % 1, 0, atol=1e-6) |
                                     np.isclose((df["bias"] * 20) % 1, 1, atol=1e-6)).mean(), 3),
    "share_multiple_of_0.10": round((np.isclose((df["bias"] * 10) % 1, 0, atol=1e-6) |
                                     np.isclose((df["bias"] * 10) % 1, 1, atol=1e-6)).mean(), 3),
}
display(pd.Series(overall, name="overall"))

In [ ]:
def r_squared(groups):
    y = df["bias"]
    fitted = y.groupby(groups).transform("mean")   # best prediction knowing only the group
    return 1 - ((y - fitted) ** 2).sum() / ((y - y.mean()) ** 2).sum()

r2 = {"outlet": round(r_squared(df["source"]), 3),
      "outlet + section": round(r_squared([df["source"], df["section"]]), 3)}
display(pd.Series(r2, name="R² of label explained by"))

sec = df.groupby(["source", "section"])["bias"].agg(n="count", mean="mean").round(3)
display(sec[sec["n"] >= 15])

FINDINGS["E2_label_distribution"] = {"per_outlet": desc.to_dict(orient="index"), "overall": overall, "r2": r2}

In [ ]:
outlets = df["source"].value_counts().index.tolist()
fig, axes = plt.subplots(len(outlets), 1, figsize=(9, 2.4 * len(outlets)), sharex=True)
bins = np.arange(0, 1.0001, 0.025)
for ax, src in zip(np.atleast_1d(axes), outlets):
    s = df.loc[df["source"] == src, "bias"]
    ax.hist(s, bins=bins, color="#4c72b0", edgecolor="white")
    ax.axvline(0.5, color="#c44e52", ls="--", lw=1)
    ax.set_title(f"{src}  (n={len(s)}, exactly 0.50: {(s.round(3)==0.5).mean():.0%})", loc="left", fontsize=10)
    ax.set_ylabel("articles")
axes[-1].set_xlabel("Gemini bias label  (0 = far-left … 1 = far-right)")
fig.suptitle("E2 — Label distribution per outlet", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT / "e2_label_hist.png", dpi=130)
plt.show()

**Read:** a spike at 0.50 + little mass above 0.70 = compressed, lopsided scale. R² ≈ 0.3 means outlet matters but doesn't decide the label, so the main defect is the **scale**, not "labels = outlet averages".

## E3 — What would a model trained on these labels learn?
**Setup:** TF-IDF + ridge (**rung 0** of the Phase 5 ladder), trained on 80% of efsyn + protothema, tested on the other 20% and on **unseen iefimerida**.

**Baselines** (a metric means nothing without one):
- `outlet mean`: a **pure outlet detector**, knows nothing about content
- `global mean`, `constant 0.5`: the "always neutral" strategy

**Key metrics:**
- **within-outlet ρ**: ranking quality *inside* one outlet. The outlet detector scores 0 here by construction, so any skill must come from content.
- **outlet-separation AUC**: if predictions separate outlets more sharply than the labels do, the model exaggerates the outlet: the shortcut in action.

**Caveat:** a random split lets the same event appear in train and test, so results are optimistic.

In [ ]:
pool = df[df["source"].isin(["efsyn.gr", "protothema.gr"])].reset_index(drop=True)
unseen = df[df["source"] == "iefimerida.gr"].reset_index(drop=True)
train, test = train_test_split(pool, test_size=0.2, stratify=pool["source"], random_state=SEED)

reg = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50_000,
                    sublinear_tf=True, strip_accents="unicode"),
    RidgeCV(alphas=np.logspace(-1, 2.5, 10)),
).fit(train["text_masked"], train["bias"])

outlet_means = train.groupby("source")["bias"].mean()
global_mean = train["bias"].mean()

def predictions(frame):
    return {
        "TF-IDF ridge (rung 0)": np.clip(reg.predict(frame["text_masked"]), 0, 1),
        "outlet mean (pure outlet detector)": frame["source"].map(outlet_means).fillna(global_mean).to_numpy(),
        "global mean": np.full(len(frame), global_mean),
        "constant 0.5": np.full(len(frame), 0.5),
    }

def spear(y, p):
    # a constant prediction can't rank anything: Spearman is undefined (shown as NaN)
    return np.nan if np.ptp(p) < 1e-12 else spearmanr(y, p).statistic

In [ ]:
rows = []
for split_name, frame in [("held-out efsyn+protothema", test), ("unseen: iefimerida", unseen)]:
    for model, p in predictions(frame).items():
        row = {"split": split_name, "model": model,
               "spearman": spear(frame["bias"], p), "MAE": mean_absolute_error(frame["bias"], p)}
        if split_name.startswith("held-out"):
            for src in ["efsyn.gr", "protothema.gr"]:
                m = (frame["source"] == src).to_numpy()
                # a constant prediction has no ranking skill: 0 by definition
                row[f"within {src}"] = spear(frame["bias"][m], p[m]) if np.ptp(p[m]) > 1e-12 else 0.0
        rows.append(row)
e3 = pd.DataFrame(rows).round(3)
display(e3)

is_proto = (test["source"] == "protothema.gr").astype(int)
p_model = predictions(test)["TF-IDF ridge (rung 0)"]
auc = {"labels separate outlets (AUC)": round(roc_auc_score(is_proto, test["bias"]), 3),
       "model predictions separate outlets (AUC)": round(roc_auc_score(is_proto, p_model), 3)}
display(pd.Series(auc))
FINDINGS["E3_generalisation"] = {"table": e3.to_dict(orient="records"), "outlet_separation": auc,
                                 "ridge_alpha": float(reg.named_steps["ridgecv"].alpha_)}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
colors = {"efsyn.gr": "#c44e52", "protothema.gr": "#4c72b0", "iefimerida.gr": "#55a868"}
for ax, (title, frame) in zip(axes, [("held-out efsyn + protothema", test), ("unseen outlet: iefimerida", unseen)]):
    p = np.clip(reg.predict(frame["text_masked"]), 0, 1)
    for src, g in frame.assign(pred=p).groupby("source"):
        ax.scatter(g["bias"], g["pred"], s=10, alpha=.5, color=colors.get(src, "grey"), label=src)
    ax.plot([0, 1], [0, 1], color="black", lw=.8, ls="--")
    ax.set(title=title, xlabel="Gemini label", xlim=(0, 1), ylim=(0, 1))
    ax.legend(fontsize=8, loc="upper left")
axes[0].set_ylabel("rung-0 prediction")
fig.suptitle("E3 — Predictions vs labels", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT / "e3_pred_vs_label.png", dpi=130)
plt.show()

**Read:**

| Pattern | Meaning |
| --- | --- |
| Good overall ρ, low within-outlet ρ | Skill is mostly outlet separation |
| Overall ρ ≈ outlet-mean baseline | Adds little beyond knowing the outlet |
| Model AUC > label AUC | Exaggerates the outlet difference |
| MAE ≈ constant 0.5 | Neutral sink dominates; weak learning signal |
| NaN ρ for constant baselines | Expected; compare those by MAE |

iefimerida labels are ~all 0.50, so a low ρ there partly means *nothing to rank*.

## E4 — Blind reading of 30 articles
**Question:** is the 0.50 pile truly neutral, or slanted articles labelled neutral? Each answer needs a different fix: a **stance gate** (Step 1.2) vs **multiple labellers + bias checks** (Steps 3.1, 3.3).

**Sample:** 15 protothema `/politics/` articles labelled exactly 0.50, plus 15 efsyn `/politiki/` articles labelled ≤ 0.20 (control).

**Fill in `outputs/e4_reading_sheet.csv`** (outlets masked, rows shuffled):

| Column | What to write |
| --- | --- |
| `has_stance` | **yes** if the article takes a side: evaluates actors ("αποτυχία", "θρίαμβος"), frames selectively, or quotes mostly one side without counterpoint. **no** if it reports what happened and who said what, neutrally or with balanced quotes. A quote's opinion is not the article's opinion. |
| `your_score_0_1` | 0 = far-left … 0.5 = centre … 1 = far-right. Empty if `has_stance = no` |
| `notes` | The phrase that decided it |

Don't open `e4_key.csv` until done. One reader is a **hint**, not proof; your own politics affect the reading.

In [ ]:
SHEET, KEY = OUT / "e4_reading_sheet.csv", OUT / "e4_key.csv"

if SHEET.exists():
    print(f"{SHEET.name} already exists, not overwriting (delete it to regenerate a fresh sample).")
else:
    proto_neutral = df[(df["source"] == "protothema.gr") & (df["section"] == "politics") & (df["bias"].round(3) == 0.5)]
    efsyn_left = df[(df["source"] == "efsyn.gr") & (df["section"] == "politiki") & (df["bias"] <= 0.2)]
    print(f"pool sizes: protothema/politics @0.50 = {len(proto_neutral)}, efsyn/politiki <=0.20 = {len(efsyn_left)}")
    sample = pd.concat([proto_neutral.sample(15, random_state=SEED), efsyn_left.sample(15, random_state=SEED)])
    sample = sample.sample(frac=1, random_state=SEED).reset_index(drop=True)      # shuffle, so groups aren't adjacent
    sample["id"] = [f"A{i+1:02d}" for i in range(len(sample))]

    sheet = sample[["id", "title_masked", "text_masked"]].rename(columns={"title_masked": "title", "text_masked": "text"})
    sheet["has_stance"], sheet["your_score_0_1"], sheet["notes"] = "", "", ""
    sheet.to_csv(SHEET, index=False, encoding="utf-8-sig")       # utf-8-sig: Excel shows Greek correctly
    sample[["id", "source", "section", "url", "bias"]].rename(columns={"bias": "gemini_bias"}).to_csv(KEY, index=False)
    print(f"wrote {SHEET} and {KEY}. Fill in the sheet before opening the key.")

In [ ]:
# Run this after filling in the reading sheet.
filled = pd.read_csv(SHEET, encoding="utf-8-sig")
filled["has_stance"] = filled["has_stance"].astype(str).str.strip().str.lower()
done = filled[filled["has_stance"].isin(["yes", "no", "y", "n", "ναι", "οχι", "όχι"])]

if len(done) < len(filled):
    print(f"{len(done)}/{len(filled)} rows filled in. Finish the sheet, then re-run this cell.")
else:
    res = done.merge(pd.read_csv(KEY), on="id")
    res["has_stance"] = res["has_stance"].isin(["yes", "y", "ναι"])
    res["your_score_0_1"] = pd.to_numeric(res["your_score_0_1"], errors="coerce")
    summary = res.groupby("source").agg(
        n=("id", "count"),
        share_with_stance=("has_stance", "mean"),
        your_mean_score=("your_score_0_1", "mean"),
        gemini_mean=("gemini_bias", "mean"),
    ).round(3)
    display(summary)
    scored = res.dropna(subset=["your_score_0_1"])
    if len(scored) > 2:
        summary_mae = round(mean_absolute_error(scored["gemini_bias"], scored["your_score_0_1"]), 3)
        print("MAE you vs Gemini (articles you scored):", summary_mae)
    FINDINGS["E4_blind_reading"] = summary.to_dict(orient="index")

**Read:** look at protothema's `share_with_stance` and `your_mean_score`. Many "neutral" articles with a clear stance, while you agree with Gemini on the efsyn control → **asymmetric labeller bias**. Mostly no stance → **missing stance gate**. Report the proportion.

## E5 — Claude vs Gemini (20 shared articles)
Tests info.txt's assumption that frontier models give similar scores. Reports r, ρ, MAE and **Krippendorff's α**, the same statistic we'll use for human agreement, so the numbers are comparable. A **bootstrap CI** shows how uncertain ρ is with n = 20.

In [ ]:
def krippendorff_alpha_interval(matrix):
    # matrix: units × raters, no missing values. Interval metric: δ(a, b) = (a − b)².
    m = np.asarray(matrix, dtype=float)
    n_units, n_raters = m.shape
    d_obs = np.mean([np.mean([(a - b) ** 2 for i, a in enumerate(row) for j, b in enumerate(row) if i != j])
                     for row in m])
    vals = m.ravel()
    diffs = (vals[:, None] - vals[None, :]) ** 2
    d_exp = diffs.sum() / (len(vals) * (len(vals) - 1))
    return 1 - d_obs / d_exp

claude = pd.DataFrame(read_any(DATA / "datasets/training/claude_labels.jsonl"))
gemini = pd.DataFrame(read_any(DATA / "datasets/training/gemini_labels.jsonl"))
for f in (claude, gemini):
    f["bias"] = f["ai_labels"].map(lambda d: d.get("bias")).astype(float)
pair = claude[["url", "title", "bias"]].merge(gemini[["url", "bias"]], on="url", suffixes=("_claude", "_gemini"))
pair["abs_diff"] = (pair["bias_claude"] - pair["bias_gemini"]).abs()

rng = np.random.default_rng(SEED)
boot = []
for _ in range(2000):
    s = pair.sample(len(pair), replace=True, random_state=int(rng.integers(1e9)))
    if s["bias_claude"].nunique() > 1 and s["bias_gemini"].nunique() > 1:
        boot.append(spearmanr(s["bias_claude"], s["bias_gemini"]).statistic)

e5 = {
    "n": len(pair),
    "pearson_r": round(pearsonr(pair["bias_claude"], pair["bias_gemini"]).statistic, 3),
    "spearman_rho": round(spearmanr(pair["bias_claude"], pair["bias_gemini"]).statistic, 3),
    "spearman_95ci": [round(float(np.percentile(boot, 2.5)), 3), round(float(np.percentile(boot, 97.5)), 3)],
    "mean_abs_diff": round(pair["abs_diff"].mean(), 3),
    "share_diff_ge_0.2": round((pair["abs_diff"] >= 0.2).mean(), 3),
    "krippendorff_alpha_interval": round(krippendorff_alpha_interval(pair[["bias_claude", "bias_gemini"]]), 3),
    "mean_claude_minus_gemini": round((pair["bias_claude"] - pair["bias_gemini"]).mean(), 3),
}
display(pd.Series(e5, name="Claude vs Gemini"))
display(pair.sort_values("abs_diff", ascending=False).head(5)[["title", "bias_claude", "bias_gemini", "abs_diff"]])
FINDINGS["E5_llm_agreement"] = e5

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 4.5))
ax.scatter(pair["bias_gemini"], pair["bias_claude"], color="#4c72b0")
ax.plot([0, 1], [0, 1], color="black", lw=.8, ls="--")
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Gemini", ylabel="Claude",
       title=f"E5 — same 20 articles (α = {e5['krippendorff_alpha_interval']})")
fig.tight_layout()
fig.savefig(OUT / "e5_claude_vs_gemini.png", dpi=130)
plt.show()

**Read:** α ≥ 0.8 reliable, ≥ 0.667 acceptable. A non-zero `mean_claude_minus_gemini` means a **systematic offset**, which is why the roadmap calibrates LLM scores against anchor articles (Step 3.2).

## Save findings
Every number in the findings note should come from this file.

In [ ]:
def to_jsonable(o):
    if isinstance(o, dict):  return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_jsonable(v) for v in o]
    if isinstance(o, (np.floating, float)): return None if np.isnan(o) else round(float(o), 4)
    if isinstance(o, np.integer): return int(o)
    return o

(OUT / "00_findings_auto.json").write_text(json.dumps(to_jsonable(FINDINGS), ensure_ascii=False, indent=2), encoding="utf-8")
print("saved", OUT / "00_findings_auto.json")
print(json.dumps(to_jsonable({k: v for k, v in FINDINGS.items() if k != "E3_generalisation"}), ensure_ascii=False, indent=1)[:3000])

## Next: `research/00_findings.md` (one page)
1. **Context** — data and why we checked (2 lines)
2. **Findings** — one line per experiment, with its number
3. **Implications** — each finding → roadmap step (2.1 outlets, 1.2 stance gate, 3.1/3.2 labellers + anchors, 2.3 gold set)
4. **Limits** — no human gold yet, optimistic random split, low-variance iefimerida, E4 = one reader